# 02 — Feature Engineering

One extractor (`src/emotion_recognition/features/extractor.py`) produces every feature that the project models consume:

| Feature | Shape (4 s @ 16 kHz) | Consumers |
|---|---|---|
| log-Mel spectrogram | `(64, 251)` | CNN / seq models |
| MFCC + delta + delta-delta | `(120, 251)` | seq / classical |
| Spectral stack (ZCR, RMS, centroid, bandwidth, rolloff, chroma ×12) | `(17, 251)` | classical |
| Pitch contour (autocorrelation F0) + prosodic vector | `(1, 251)` / `(8,)` | classical / prosody analysis |
| Statistics-pooled utterance vector (mean/std per group) | `(282,)` | sklearn baselines |

Fixed 4 s window (pad/crop at the end), 16 kHz mono, `n_fft=1024`, `hop=256`. The extractor is the *single* audio→feature code path, so training and serving always agree.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make the package importable whether the kernel starts at the repo root
# or inside notebooks/.
ROOT = Path.cwd()
while not (ROOT / "src" / "emotion_recognition").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
from emotion_recognition.features.extractor import (
    DEFAULT_GROUPS,
    FeatureConfig,
    FeatureExtractor,
)

extractor = FeatureExtractor(FeatureConfig())
print("feature groups:", DEFAULT_GROUPS)
print(f"window: {extractor.cfg.duration_s:g} s @ {extractor.cfg.target_sr} Hz = {extractor.cfg.n_samples()} samples")
print(f"frames: {extractor.cfg.n_frames()}")

## Extract a sample of utterances and verify shapes

Uses the prepared metadata (if present) to pull a handful of files per split/emotion for downstream plotting.

In [ ]:
metadata_path = ROOT / "data" / "processed" / "metadata.csv"
assert metadata_path.is_file(), f"{metadata_path} not found. Run `make prepare` first."

metadata = pd.read_csv(metadata_path)
emotions = sorted(metadata["emotion"].unique())
print("emotions:", emotions)
print(f"utterances: {len(metadata):,}")

In [ ]:
EMOTION_COLORS = {
    "neutral":        "#9aa0a6",
    "calm":           "#a5d8ff",
    "happy":          "#ffe083",
    "sad":            "#8f9df0",
    "angry":          "#ff6b6b",
    "fearful":        "#b197fc",
    "disgust":        "#69db7c",
    "surprised":      "#ffa94d",
}

def sample_one_per_emotion(meta: pd.DataFrame) -> pd.DataFrame:
    return meta.groupby("emotion").sample(1, random_state=42)

picked = sample_one_per_emotion(metadata)
picked[["file_name", "emotion", "dataset_split", "speaker_id"]].assign(path=(
    picked["file_name"]
))

In [ ]:
bundles = {}
for row in picked.itertuples(index=False):
    path = ROOT / "data" / "raw" / "ravdess" / row.file_name
    if path.is_file():
        bundles[row.emotion] = extractor.extract(path)

print(f"extracted {len(bundles)}/{len(picked)} utterances")
if bundles:
    b = next(iter(bundles.values()))
    print(f"log_mel  {tuple(b.log_mel.shape)}")
    print(f"mfcc     {tuple(b.mfcc_stack.shape)}")
    print(f"spectral {tuple(b.spectral.shape)}")
    print(f"pitch    {tuple(b.pitch.shape)}")
    print(f"prosodic {tuple(b.prosodic.shape)}")

## What the model sees: log-Mel spectrograms per emotion

The deep model consumes these `(64, 251)` matrices. Compare structure — e.g. *angry* vs *calm* harmonic excitation, formant crowding, and overall loudness.

In [ ]:
if bundles:
    n = len(bundles)
    fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(15, 6), squeeze=False)
    axes = axes.ravel()
    for ax, (emotion, b) in zip(axes, bundles.items()):
        img = b.log_mel
        ax.imshow(
            img,
            aspect="auto",
            origin="lower",
            cmap="magma",
            extent=[0, b.waveform.size / b.sample_rate, 0, extractor.cfg.fmax],
        )
        ax.set_title(emotion, color=EMOTION_COLORS[emotion])
        ax.set_xlabel("time (s)")
        ax.set_ylabel("Hz")
    fig.suptitle("log-Mel spectrograms (one file per emotion)")
    fig.tight_layout()
    plt.show()
else:
    print("no files extracted; rerun after make prepare")

## The classical-BL vector: statistics pooling

For sklearn baselines the frame matrices are collapsed to mean/std per group, giving one fixed-length vector (`282` dims by default). This cell shows the per-group contribution.

In [ ]:
from emotion_recognition.features.extractor import pool_utterance

if bundles:
    b = next(iter(bundles.values()))
    v = pool_utterance(b, extractor.cfg)
    print(f"pooled vector length: {v.size}")
    n_features = {
        "mfcc": b.mfcc.shape[0],
        "delta": b.mfcc_delta.shape[0],
        "delta2": b.mfcc_delta2.shape[0],
        "spectral": b.spectral.shape[0],
        "prosodic": b.prosodic.size,
    }
    for group in DEFAULT_GROUPS:
        dims = n_features[group] * len(extractor.cfg.pool_stats)
        print(f"  {group:<9} {dims:>3} dims" if group != "prosodic" else f"  {group:<9} {dims:>3} dims (pre-aggregated)")
else:
    print("no files extracted; rerun after make prepare")

## Prosodic statistics by emotion

Voicing and pitch contours carry a lot of affect signal. Plot two summary stats over the whole (or sampled) dataset: voiced-F0 mean and voiced ratio. Light sampling keeps the notebook fast.

In [ ]:
SAMPLE_PER_EMOTION = 20  # None = all rows

def build_prosody_frame(meta: pd.DataFrame, per_emotion: int | None) -> pd.DataFrame:
    if per_emotion is not None:
        meta = meta.groupby("emotion").sample(per_emotion, random_state=42)
    rows = []
    for r in meta.itertuples(index=False):
        path = ROOT / "data" / "raw" / "ravdess" / r.file_name
        if path.is_file():
            b = extractor.extract(path)
            voiced = b.pitch[0]
            voiced = voiced[voiced > 0]
            rows.append(
                {
                    "emotion": r.emotion,
                    "f0_mean": voiced.mean() if voiced.size else np.nan,
                    "voiced_ratio": b.prosodic[4],
                    "rms": b.prosodic[5],
                }
            )
    return pd.DataFrame(rows)

In [ ]:
prosody = build_prosody_frame(metadata, SAMPLE_PER_EMOTION)
print(prosody.groupby("emotion")[["f0_mean", "voiced_ratio", "rms"]].mean().round(2).to_string())

In [ ]:
if not prosody.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for i, col in enumerate(("f0_mean", "voiced_ratio")):
        axes[i].boxplot(
            [prosody.loc[prosody["emotion"] == e, col].dropna() for e in emotions],
            tick_labels=emotions,
        )
        axes[i].set_xticklabels(emotions, rotation=45, ha="right")
        axes[i].set_title(f"{col} by emotion")
    fig.tight_layout()
    plt.show()

## Interpretation checklist

Questions to answer once real RAVDESS data is processed (keep these conclusions alongside experiment results in `reports/`):

- **Spectrograms:** which emotions exhibit the most/least harmonic structure? Do *angry* files show higher overall loudness (bright bins across all mels)?
- **Pooled vector:** is 282 dims with ~40k train samples a reasonable aspect ratio for a linear classifier, or should groups be ablated?
- **Prosody:** is *surprised* wider in pitch range? Is *sad* lower / less voiced? Are F0 stats enough to separate *calm* vs *neutral*?
- **Ablation plan:** which single group (MFCC vs spectral vs prosodic) looks most separable — this focuses which baselines to prioritize in Phase 4.

In [ ]:
# Sanity checks that keep every pipeline run honest.
b = extractor.from_waveform(np.zeros(32000, dtype=np.float32), extractor.cfg.target_sr)
assert b.log_mel.shape == (extractor.cfg.n_mels, 251)
assert (b.pitch == 0).all(), "silence must be unvoiced"
assert pool_utterance(b, extractor.cfg).size == 282
print("extractor sanity checks OK")